# Phase 2 Notebook 03G: Signal Diversity Engine

Purpose: add a diversity-aware signal selection layer after Notebook 03F and before Alpha Construction. This notebook identifies high-quality signals that are less redundant with each other, so later alpha construction can avoid being dominated by highly correlated variants.

Scope boundaries:
- This notebook does not create new signals.
- It does not change signal formulas, scoring, WFV, decay, regime, health, reproducibility, alpha construction, stress, freeze, portfolio, or ML logic.
- It writes only `signal_diversity_*` diagnostic and selection tables.


## 1. Imports and Config

In [1]:
from pathlib import Path
import os
import sqlite3
import sys
import time
from contextlib import contextmanager

import psutil

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import ensure_candidate_signal_indexes, get_db_path
from src.run_config import make_run_id, make_run_timestamp
from src.signal_diversity import (
    build_cluster_diversity_report,
    build_diversity_candidate_table,
    build_family_diversity_report,
    build_signal_panels,
    build_signal_similarity_matrix,
    compute_diversity_diagnostics,
    greedy_diversity_selection,
    load_candidate_signal_rows,
    load_signal_diversity_inputs,
)
from src.signal_diversity_storage import SIGNAL_DIVERSITY_TABLES, save_signal_diversity_outputs
from src.signal_storage import load_candidate_signals_by_names

DB_PATH = get_db_path()
DIVERSITY_VERSION = "phase2_signal_diversity_v1"
CORRELATION_THRESHOLD = 0.85
MIN_SELECTED = 3
INCLUDE_WATCHLIST = False
INCLUDE_ORTHOGONAL_DIVERSIFIERS = True
ORTHOGONAL_DIVERSIFIER_VERSION = "phase2_orthogonal_signals_v2"
ORTHOGONAL_DIVERSIFIER_MIN_HEALTH_SCORE = 60
ORTHOGONAL_DIVERSIFIER_MIN_PASS_RATE = 0.60

pd.set_option("display.max_columns", 200)
DB_PATH


def readback_sql(query, conn, params=None, **kwargs):
    normalized = " ".join(query.lower().split())
    reads_candidate_signals = "from candidate_signals_current" in normalized or 'from "candidate_signals_current"' in normalized
    has_signal_filter = "where" in normalized and "signal_name" in normalized
    is_aggregate_count = "count(" in normalized and "group by" in normalized
    if reads_candidate_signals and not has_signal_filter and not is_aggregate_count:
        raise ValueError(
            "Unsafe 03G readback query blocked: candidate_signals_current reads must "
            "filter by signal_name, or be aggregate COUNT/GROUP BY diagnostics."
        )
    return pd.read_sql_query(query, conn, params=params, **kwargs)

PROFILE_03G = True
_profile_process = psutil.Process(os.getpid())
profile_records = []

def _profile_memory_mb():
    return _profile_process.memory_info().rss / (1024 ** 2)

def _profile_print(record):
    detail_parts = [
        f"{key}={value}"
        for key, value in record.items()
        if key not in {"block", "elapsed_seconds", "memory_before_mb", "memory_after_mb", "memory_delta_mb"}
    ]
    details = " | " + " | ".join(detail_parts) if detail_parts else ""
    print(
        f"[03G_PROFILE] {record['block']} | "
        f"elapsed_seconds={record['elapsed_seconds']:.3f} | "
        f"memory_before_mb={record['memory_before_mb']:.1f} | "
        f"memory_after_mb={record['memory_after_mb']:.1f} | "
        f"memory_delta_mb={record['memory_delta_mb']:.1f}"
        f"{details}"
    )

@contextmanager
def profile_block(block_name):
    metrics = {}
    if not PROFILE_03G:
        yield metrics
        return
    start_memory = _profile_memory_mb()
    start_time = time.perf_counter()
    try:
        yield metrics
    finally:
        end_time = time.perf_counter()
        end_memory = _profile_memory_mb()
        record = {
            "block": block_name,
            "elapsed_seconds": end_time - start_time,
            "memory_before_mb": start_memory,
            "memory_after_mb": end_memory,
            "memory_delta_mb": end_memory - start_memory,
        }
        record.update(metrics)
        profile_records.append(record)
        _profile_print(record)

DB_PATH


PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 2. Create run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase2_signal_diversity")
run_timestamp = make_run_timestamp()

run_id, run_timestamp

('phase2_signal_diversity_20260511_172132', '2026-05-11 17:21:32')

## 3. Load health and reproducibility inputs

In [3]:
with profile_block("input table loading") as profile_metrics:
    inputs = load_signal_diversity_inputs(db_path=DB_PATH)

    input_shapes = pd.DataFrame(
        [
            {"input_name": name, "n_rows": len(df), "n_columns": len(df.columns)}
            for name, df in inputs.items()
        ]
    )
    profile_metrics["input_tables_loaded"] = len(inputs)
    profile_metrics["input_rows_loaded"] = int(input_shapes["n_rows"].sum())
    profile_metrics["health_rows"] = len(inputs["health"])
    profile_metrics["reproducibility_gate_rows"] = len(inputs["reproducibility_gate"])
    display(input_shapes)

    # Indexes keep 03G diagnostics on indexed signal-level scans instead of broad joins.
    candidate_signal_indexes = ensure_candidate_signal_indexes(DB_PATH)
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS idx_signal_best_horizon_current_signal
            ON signal_best_horizon_current(signal_name)
            """
        )
        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS idx_candidate_signal_quality_gate_current_signal
            ON candidate_signal_quality_gate_current(signal_name)
            """
        )
        conn.commit()
        candidate_best_horizon_gap = readback_sql(
            """
            WITH candidate_signal_counts AS (
                SELECT
                    signal_name,
                    COUNT(*) AS candidate_rows,
                    COUNT(DISTINCT Date) AS n_dates
                FROM candidate_signals_current
                GROUP BY signal_name
            ),
            best_horizon_signals AS (
                SELECT DISTINCT signal_name
                FROM signal_best_horizon_current
            )
            SELECT
                c.signal_name,
                COALESCE(q.status, 'MISSING_QUALITY_GATE') AS quality_gate_status,
                q.missing_pct,
                q.finite_pct,
                c.candidate_rows,
                c.n_dates,
                CASE WHEN b.signal_name IS NULL THEN 0 ELSE 1 END AS best_horizon_rows
            FROM candidate_signal_counts c
            LEFT JOIN best_horizon_signals b
                ON b.signal_name = c.signal_name
            LEFT JOIN candidate_signal_quality_gate_current q
                ON q.signal_name = c.signal_name
            WHERE b.signal_name IS NULL
            ORDER BY quality_gate_status, c.signal_name
            """,
            conn,
        )
    profile_metrics["candidate_signal_indexes"] = len(candidate_signal_indexes)
    profile_metrics["best_horizon_gap_rows"] = len(candidate_best_horizon_gap)

    if not candidate_best_horizon_gap.empty:
        print(
            "Signal exists in candidate_signals_current but has no best-horizon scoring row, "
            "so it is excluded from diversity analysis."
        )
        display(candidate_best_horizon_gap.head(20))


,input_name,n_rows,n_columns
0,health,92,31
1,reproducibility_gate,1,19


Signal exists in candidate_signals_current but has no best-horizon scoring row, so it is excluded from diversity analysis.


,signal_name,quality_gate_status,missing_pct,finite_pct,candidate_rows,n_dates,best_horizon_rows
0,amihud_illiq_20,REJECTED_DATA_QUALITY,0.438266,0.561734,1002844,2098,0
1,breakout_20,REJECTED_DATA_QUALITY,0.437856,0.562144,1002844,2098,0
2,breakout_60,REJECTED_DATA_QUALITY,0.494885,0.505115,1002844,2098,0
3,breakout_up_20,REJECTED_DATA_QUALITY,0.468612,0.531388,1002844,2098,0
4,breakout_up_60,REJECTED_DATA_QUALITY,0.543075,0.456925,1002844,2098,0
5,disc_corr_change_s5_l60_rank,REJECTED_DATA_QUALITY,0.381009,0.618991,1002844,2098,0
6,disc_corr_change_s5_l60_raw,REJECTED_DATA_QUALITY,0.381024,0.618976,1002844,2098,0
7,disc_corr_change_s5_l60_wz,REJECTED_DATA_QUALITY,0.381024,0.618976,1002844,2098,0
8,disc_corr_change_s5_l60_z,REJECTED_DATA_QUALITY,0.381024,0.618976,1002844,2098,0
9,disc_liq_adj_ret_w5_dollarvolumerank_rank,REJECTED_DATA_QUALITY,0.393018,0.606982,1002844,2098,0


[03G_PROFILE] input table loading | elapsed_seconds=7.385 | memory_before_mb=144.5 | memory_after_mb=116.5 | memory_delta_mb=-28.0 | input_tables_loaded=2 | input_rows_loaded=93 | health_rows=92 | reproducibility_gate_rows=1 | candidate_signal_indexes=3 | best_horizon_gap_rows=85


## 4. Build diversity candidate universe

In [4]:
with profile_block("eligible candidate filtering") as profile_metrics:
    diversity_candidates = build_diversity_candidate_table(
        health=inputs["health"],
        reproducibility_gate=inputs["reproducibility_gate"],
        include_watchlist=INCLUDE_WATCHLIST,
        include_orthogonal_diversifiers=INCLUDE_ORTHOGONAL_DIVERSIFIERS,
        orthogonal_diversifier_version=ORTHOGONAL_DIVERSIFIER_VERSION,
        orthogonal_diversifier_min_health_score=ORTHOGONAL_DIVERSIFIER_MIN_HEALTH_SCORE,
        orthogonal_diversifier_min_pass_rate=ORTHOGONAL_DIVERSIFIER_MIN_PASS_RATE,
    )

    eligible_candidates = diversity_candidates.loc[diversity_candidates["eligible_for_diversity"].eq(True)].copy()
    eligible_signal_names = eligible_candidates["signal_name"].dropna().astype(str).unique().tolist()

    candidate_tier_counts = (
        diversity_candidates["diversity_candidate_tier"]
        .value_counts(dropna=False)
        .rename_axis("diversity_candidate_tier")
        .reset_index(name="n_candidates")
    )
    orthogonal_diversifier_candidates = diversity_candidates.loc[
        diversity_candidates["diversity_candidate_tier"].eq("ORTHOGONAL_DIVERSIFIER")
    ].copy()
    range_expansion_candidates = diversity_candidates.loc[
        diversity_candidates["signal_name"].eq("range_expansion_failure_5")
    ].copy()

    profile_metrics["candidate_rows"] = len(diversity_candidates)
    profile_metrics["eligible_candidate_rows"] = len(eligible_candidates)
    profile_metrics["unique_eligible_signals"] = len(eligible_signal_names)

    print(f"Candidate rows from reproducibility gate: {len(diversity_candidates)}")
    print(f"Eligible diversity candidates: {len(eligible_candidates)}")
    print(f"Unique eligible signal names: {len(eligible_signal_names)}")
    print("Candidate counts by diversity_candidate_tier")
    display(candidate_tier_counts)
    print("Orthogonal diversifier candidates admitted")
    display(orthogonal_diversifier_candidates)
    print("range_expansion_failure_5 candidate rows")
    display(range_expansion_candidates)
    display(diversity_candidates)


Candidate rows from reproducibility gate: 1
Eligible diversity candidates: 1
Unique eligible signal names: 1
Candidate counts by diversity_candidate_tier


,diversity_candidate_tier,n_candidates
0,ORTHOGONAL_DIVERSIFIER,1


Orthogonal diversifier candidates admitted


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version,signal_direction,signal_strength,recommended_use,regime_fragility_flag,decay_risk_flag,scoring_status,decay_status,diversity_candidate_tier,eligible_for_diversity
0,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,phase2_orthogonal_signals_v2,62.0,WATCHLIST_RESEARCH,14,12,0.857143,0.015966,0.00472,CONDITIONAL_PASS,WATCHLIST_ALPHA_RESEARCH,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1,POSITIVE_EDGE,WEAK,CONDITIONAL,HIGH_REGIME_FRAGILITY,LOW_DECAY_RISK,WATCHLIST,STABLE,ORTHOGONAL_DIVERSIFIER,True


range_expansion_failure_5 candidate rows


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version,signal_direction,signal_strength,recommended_use,regime_fragility_flag,decay_risk_flag,scoring_status,decay_status,diversity_candidate_tier,eligible_for_diversity


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate,run_id,reproducibility_version,signal_direction,signal_strength,recommended_use,regime_fragility_flag,decay_risk_flag,scoring_status,decay_status,diversity_candidate_tier,eligible_for_diversity
0,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,phase2_orthogonal_signals_v2,62.0,WATCHLIST_RESEARCH,14,12,0.857143,0.015966,0.00472,CONDITIONAL_PASS,WATCHLIST_ALPHA_RESEARCH,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1,POSITIVE_EDGE,WEAK,CONDITIONAL,HIGH_REGIME_FRAGILITY,LOW_DECAY_RISK,WATCHLIST,STABLE,ORTHOGONAL_DIVERSIFIER,True


[03G_PROFILE] eligible candidate filtering | elapsed_seconds=0.021 | memory_before_mb=118.1 | memory_after_mb=121.2 | memory_delta_mb=3.2 | candidate_rows=1 | eligible_candidate_rows=1 | unique_eligible_signals=1


## 5. Load candidate signal panels

In [5]:
with profile_block("signal panel loading") as profile_metrics:
    candidate_signal_rows = load_candidate_signal_rows(eligible_signal_names, db_path=DB_PATH)
    signal_panels = build_signal_panels(
        candidate_table=diversity_candidates,
        candidate_signals_long=candidate_signal_rows,
    )

    panel_summary = pd.DataFrame(
        [
            {
                "signal_key": signal_key,
                "n_dates": panel.shape[0],
                "n_tickers": panel.shape[1],
                "finite_pct": float(panel.notna().to_numpy().mean()) if panel.size else 0.0,
            }
            for signal_key, panel in signal_panels.items()
        ]
    )

    profile_metrics["candidate_signal_rows_loaded"] = len(candidate_signal_rows)
    profile_metrics["signal_panels_built"] = len(signal_panels)
    profile_metrics["panel_cells"] = int(sum(panel.size for panel in signal_panels.values()))

    print(f"Loaded candidate signal rows: {len(candidate_signal_rows):,}")
    print(f"Built signal panels: {len(signal_panels)}")
    display(panel_summary)


Loaded candidate signal rows: 1,002,844
Built signal panels: 1


,signal_key,n_dates,n_tickers,finite_pct
0,vol_of_vol_20__h10,2098,478,0.880582


[03G_PROFILE] signal panel loading | elapsed_seconds=1.285 | memory_before_mb=121.3 | memory_after_mb=592.3 | memory_delta_mb=471.0 | candidate_signal_rows_loaded=1002844 | signal_panels_built=1 | panel_cells=1002844


## 6. Build similarity matrix and diagnostics

In [6]:
with profile_block("similarity / correlation calculation") as profile_metrics:
    signal_diversity_similarity = build_signal_similarity_matrix(
        candidate_table=diversity_candidates,
        signal_panels=signal_panels,
    )
    signal_diversity_diagnostics = compute_diversity_diagnostics(
        similarity=signal_diversity_similarity,
        candidate_table=diversity_candidates,
    )

    n_panels = len(signal_panels)
    profile_metrics["candidate_signals"] = len(eligible_signal_names)
    profile_metrics["signal_panels"] = n_panels
    profile_metrics["pairwise_comparisons_executed"] = len(signal_diversity_similarity)
    profile_metrics["unique_off_diagonal_pairs"] = n_panels * (n_panels - 1) // 2
    profile_metrics["similarity_rows"] = len(signal_diversity_similarity)
    profile_metrics["diagnostic_rows"] = len(signal_diversity_diagnostics)

    print("Diversity diagnostics")
    display(signal_diversity_diagnostics)

    display(signal_diversity_similarity.sort_values("correlation", key=lambda s: s.abs(), ascending=False).head(20))


Diversity diagnostics


,n_candidates,avg_abs_correlation,max_abs_correlation,median_abs_correlation,effective_signal_count
0,1,NaN,NaN,NaN,1.0


,signal_key_1,signal_key_2,signal_name_1,horizon_1,signal_name_2,horizon_2,correlation
0,vol_of_vol_20__h10,vol_of_vol_20__h10,vol_of_vol_20,10,vol_of_vol_20,10,1.0


[03G_PROFILE] similarity / correlation calculation | elapsed_seconds=0.107 | memory_before_mb=584.7 | memory_after_mb=645.7 | memory_delta_mb=61.0 | candidate_signals=1 | signal_panels=1 | pairwise_comparisons_executed=1 | unique_off_diagonal_pairs=0 | similarity_rows=1 | diagnostic_rows=1


## 7. Greedy diversity selection

In [7]:
with profile_block("diversity selection") as profile_metrics:
    signal_diversity_selection = greedy_diversity_selection(
        candidate_table=diversity_candidates,
        similarity=signal_diversity_similarity,
        correlation_threshold=CORRELATION_THRESHOLD,
        min_selected=MIN_SELECTED,
    )

    selected_signals = signal_diversity_selection.loc[signal_diversity_selection["selected_flag"].eq(1)].copy()
    redundant_rejected_signals = signal_diversity_selection.loc[
        signal_diversity_selection["diversity_group"].eq("REDUNDANT_REJECTED")
    ].copy()
    forced_selected_signals = signal_diversity_selection.loc[
        signal_diversity_selection["diversity_group"].eq("FORCED_MIN_SELECTED")
    ].copy()
    orthogonal_selection_rows = signal_diversity_selection.loc[
        signal_diversity_selection["diversity_candidate_tier"].eq("ORTHOGONAL_DIVERSIFIER")
    ].copy()
    range_expansion_selection_rows = signal_diversity_selection.loc[
        signal_diversity_selection["signal_name"].eq("range_expansion_failure_5")
    ].copy()

    selection_group_counts = (
        signal_diversity_selection["diversity_group"]
        .value_counts(dropna=False)
        .rename_axis("diversity_group")
        .reset_index(name="n_signals")
    )

    selection_tier_counts = (
        signal_diversity_selection["diversity_candidate_tier"]
        .value_counts(dropna=False)
        .rename_axis("diversity_candidate_tier")
        .reset_index(name="n_signals")
    )

    profile_metrics["selection_rows"] = len(signal_diversity_selection)
    profile_metrics["selected_count"] = len(selected_signals)
    profile_metrics["redundant_rejected_count"] = len(redundant_rejected_signals)

    if len(selected_signals) < MIN_SELECTED:
        print(f"Selected signals below MIN_SELECTED={MIN_SELECTED}; only {len(selected_signals)} eligible non-redundant candidates were available.")
    elif not forced_selected_signals.empty:
        print("MIN_SELECTED was reached by force-selecting the next best core candidates after threshold relaxation.")

    print(f"Selected signals: {len(selected_signals)}")
    print("Selection group counts")
    display(selection_group_counts)
    print("Selection tier counts")
    display(selection_tier_counts)
    print("Selected signals including tier/source/cluster")
    display(selected_signals)
    print("Orthogonal diversifier selection rows")
    display(orthogonal_selection_rows)
    print("range_expansion_failure_5 selection rows")
    display(range_expansion_selection_rows)
    print("Rejected redundant signals")
    display(redundant_rejected_signals)
    display(signal_diversity_selection)


Selected signals below MIN_SELECTED=3; only 1 eligible non-redundant candidates were available.
Selected signals: 1
Selection group counts


,diversity_group,n_signals
0,ORTHOGONAL_DIVERSIFIER_SELECTED,1


Selection tier counts


,diversity_candidate_tier,n_signals
0,ORTHOGONAL_DIVERSIFIER,1


Selected signals including tier/source/cluster


...

,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group


Rejected redundant signals


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group
0,vol_of_vol_20__h10,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_DIVERSIFIER,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,62.0,WATCHLIST_ALPHA_RESEARCH,CONDITIONAL_PASS,0.857143,0.015966,1,1,0.0,Selected within correlation threshold 0.85.,ORTHOGONAL_DIVERSIFIER_SELECTED


[03G_PROFILE] diversity selection | elapsed_seconds=0.016 | memory_before_mb=644.1 | memory_after_mb=642.6 | memory_delta_mb=-1.6 | selection_rows=1 | selected_count=1 | redundant_rejected_count=0


## 8. Family diversity report

In [8]:
with profile_block("diversity report building") as profile_metrics:
    signal_diversity_family_report = build_family_diversity_report(
        candidate_table=diversity_candidates,
        selection=signal_diversity_selection,
        similarity=signal_diversity_similarity,
    )

    signal_diversity_cluster_report = build_cluster_diversity_report(
        candidate_table=diversity_candidates,
        selection=signal_diversity_selection,
        similarity=signal_diversity_similarity,
    )

    profile_metrics["family_report_rows"] = len(signal_diversity_family_report)
    profile_metrics["cluster_report_rows"] = len(signal_diversity_cluster_report)

    print("Family diversity report")
    display(signal_diversity_family_report)
    print("Cluster diversity report")
    display(signal_diversity_cluster_report)


Family diversity report


,signal_family,n_candidates,n_selected,avg_health_score,max_health_score,avg_abs_corr_within_family
0,volatility_structure,1,1,62.0,62.0,NaN


Cluster diversity report


,orthogonal_cluster,n_candidates,n_selected,n_orthogonal_diversifiers,avg_health_score,max_health_score,avg_abs_corr_within_cluster
0,volatility_structure,1,1,1,62.0,62.0,NaN


[03G_PROFILE] diversity report building | elapsed_seconds=0.006 | memory_before_mb=642.6 | memory_after_mb=642.6 | memory_delta_mb=0.0 | family_report_rows=1 | cluster_report_rows=1


## 9. Save outputs to SQLite

In [9]:
with profile_block("SQLite writes") as profile_metrics:
    saved_paths = save_signal_diversity_outputs(
        similarity=signal_diversity_similarity,
        diagnostics=signal_diversity_diagnostics,
        selection=signal_diversity_selection,
        family_report=signal_diversity_family_report,
        cluster_report=signal_diversity_cluster_report,
        db_path=DB_PATH,
        run_id=run_id,
        diversity_version=DIVERSITY_VERSION,
    )

    sqlite_tables_written = pd.DataFrame(
        [
            {
                "artifact": artifact,
                "current_table": tables[0],
                "history_table": tables[1],
                "sqlite_path": str(saved_paths[artifact]),
            }
            for artifact, tables in SIGNAL_DIVERSITY_TABLES.items()
        ]
    )

    profile_metrics["similarity_rows_written"] = len(signal_diversity_similarity)
    profile_metrics["diagnostic_rows_written"] = len(signal_diversity_diagnostics)
    profile_metrics["selection_rows_written"] = len(signal_diversity_selection)
    profile_metrics["family_report_rows_written"] = len(signal_diversity_family_report)
    profile_metrics["cluster_report_rows_written"] = len(signal_diversity_cluster_report)
    profile_metrics["sqlite_tables_written"] = len(sqlite_tables_written) * 2

    display(sqlite_tables_written)


,artifact,current_table,history_table,sqlite_path
0,similarity,signal_diversity_similarity_current,signal_diversity_similarity_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,diagnostics,signal_diversity_diagnostics_current,signal_diversity_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,selection,signal_diversity_selection_current,signal_diversity_selection_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,family_report,signal_diversity_family_report_current,signal_diversity_family_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,cluster_report,signal_diversity_cluster_report_current,signal_diversity_cluster_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


[03G_PROFILE] SQLite writes | elapsed_seconds=0.020 | memory_before_mb=642.7 | memory_after_mb=643.1 | memory_delta_mb=0.4 | similarity_rows_written=1 | diagnostic_rows_written=1 | selection_rows_written=1 | family_report_rows_written=1 | cluster_report_rows_written=1 | sqlite_tables_written=10


## 10. Final summary

In [10]:
with profile_block("display/readback sections") as profile_metrics:
    final_summary = pd.DataFrame(
        [
            {"metric": "run_id", "value": run_id},
            {"metric": "run_timestamp", "value": run_timestamp},
            {"metric": "diversity_version", "value": DIVERSITY_VERSION},
            {"metric": "candidate_count", "value": len(eligible_candidates)},
            {"metric": "selected_count", "value": int(signal_diversity_selection["selected_flag"].sum())},
            {"metric": "orthogonal_diversifier_candidates", "value": len(orthogonal_diversifier_candidates)},
            {"metric": "orthogonal_diversifier_selected", "value": int(selected_signals["diversity_candidate_tier"].eq("ORTHOGONAL_DIVERSIFIER").sum())},
            {"metric": "redundant_rejected_count", "value": len(redundant_rejected_signals)},
        ]
    )

    profile_metrics["summary_rows_displayed"] = len(final_summary)
    profile_metrics["selection_rows_displayed"] = len(signal_diversity_selection)
    profile_metrics["similarity_rows_available"] = len(signal_diversity_similarity)

    print("Candidate count")
    display(final_summary)

    print("Candidate counts by diversity_candidate_tier")
    display(candidate_tier_counts)

    print("Orthogonal diversifier candidates admitted")
    display(orthogonal_diversifier_candidates)

    print("Diversity diagnostics")
    display(signal_diversity_diagnostics)

    print("Selected signals")
    display(selected_signals)

    print("Rows for range_expansion_failure_5")
    display(range_expansion_selection_rows)

    print("Redundant rejected signals")
    display(redundant_rejected_signals)

    print("Family diversity report")
    display(signal_diversity_family_report)

    print("Cluster diversity report")
    display(signal_diversity_cluster_report)

    print("SQLite tables written")
    display(sqlite_tables_written)

    profiling_summary = pd.DataFrame(profile_records)
    if not profiling_summary.empty:
        top_bottleneck = profiling_summary.sort_values("elapsed_seconds", ascending=False).iloc[0]
        print("03G profiling summary")
        display(profiling_summary)
        print(
            f"Top bottleneck: {top_bottleneck['block']} "
            f"({top_bottleneck['elapsed_seconds']:.3f} seconds, "
            f"memory_delta_mb={top_bottleneck['memory_delta_mb']:.1f})"
        )


Candidate count


,metric,value
0,run_id,phase2_signal_diversity_20260511_172132
1,run_timestamp,2026-05-11 17:21:32
2,diversity_version,phase2_signal_diversity_v1
3,candidate_count,1
4,selected_count,1
5,orthogonal_diversifier_candidates,1
6,orthogonal_diversifier_selected,1
7,redundant_rejected_count,0


Candidate counts by diversity_candidate_tier


,diversity_candidate_tier,n_candidates
0,ORTHOGONAL_DIVERSIFIER,1


Orthogonal diversifier candidates admitted


...

SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,similarity,signal_diversity_similarity_current,signal_diversity_similarity_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,diagnostics,signal_diversity_diagnostics_current,signal_diversity_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,selection,signal_diversity_selection_current,signal_diversity_selection_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,family_report,signal_diversity_family_report_current,signal_diversity_family_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,cluster_report,signal_diversity_cluster_report_current,signal_diversity_cluster_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


03G profiling summary


,block,elapsed_seconds,memory_before_mb,memory_after_mb,memory_delta_mb,input_tables_loaded,input_rows_loaded,health_rows,reproducibility_gate_rows,candidate_signal_indexes,best_horizon_gap_rows,candidate_rows,eligible_candidate_rows,unique_eligible_signals,candidate_signal_rows_loaded,signal_panels_built,panel_cells,candidate_signals,signal_panels,pairwise_comparisons_executed,unique_off_diagonal_pairs,similarity_rows,diagnostic_rows,selection_rows,selected_count,redundant_rejected_count,family_report_rows,cluster_report_rows,similarity_rows_written,diagnostic_rows_written,selection_rows_written,family_report_rows_written,cluster_report_rows_written,sqlite_tables_written
0,input table loading,7.384892,144.453125,116.484375,-27.968750,2.0,93.0,92.0,1.0,3.0,85.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,eligible candidate filtering,0.020820,118.062500,121.218750,3.156250,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,signal panel loading,1.284967,121.312500,592.296875,470.984375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1002844.0,1.0,1002844.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,similarity / correlation calculation,0.106850,584.671875,645.703125,61.031250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,diversity selection,0.016404,644.140625,642.562500,-1.578125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,diversity report building,0.006478,642.578125,642.625000,0.046875,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
6,SQLite writes,0.019982,642.656250,643.062500,0.406250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,10.0


Top bottleneck: input table loading (7.385 seconds, memory_delta_mb=-28.0)
[03G_PROFILE] display/readback sections | elapsed_seconds=0.023 | memory_before_mb=643.1 | memory_after_mb=643.2 | memory_delta_mb=0.1 | summary_rows_displayed=8 | selection_rows_displayed=1 | similarity_rows_available=1


In [11]:
with profile_block("SQLite readback: diversity outputs") as profile_metrics:
    from src.db import load_table

    sel = load_table("signal_diversity_selection_current")
    diag = load_table("signal_diversity_diagnostics_current")
    fam = load_table("signal_diversity_family_report_current")
    cluster = load_table("signal_diversity_cluster_report_current")

    profile_metrics["selection_readback_rows"] = len(sel)
    profile_metrics["diagnostics_readback_rows"] = len(diag)
    profile_metrics["family_readback_rows"] = len(fam)
    profile_metrics["cluster_readback_rows"] = len(cluster)

    display(diag)
    display(sel.sort_values(["selected_flag", "selection_rank"], ascending=[False, True]))
    display(fam)
    display(cluster)


,n_candidates,avg_abs_correlation,max_abs_correlation,median_abs_correlation,effective_signal_count,run_id,diversity_version
0,1,None,None,None,1.0,phase2_signal_diversity_20260511_172132,phase2_signal_diversity_v1


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group,run_id,diversity_version
0,vol_of_vol_20__h10,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_DIVERSIFIER,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,62.0,WATCHLIST_ALPHA_RESEARCH,CONDITIONAL_PASS,0.857143,0.015966,1,1,0.0,Selected within correlation threshold 0.85.,ORTHOGONAL_DIVERSIFIER_SELECTED,phase2_signal_diversity_20260511_172132,phase2_signal_diversity_v1


,signal_family,n_candidates,n_selected,avg_health_score,max_health_score,avg_abs_corr_within_family,run_id,diversity_version
0,volatility_structure,1,1,62.0,62.0,None,phase2_signal_diversity_20260511_172132,phase2_signal_diversity_v1


,orthogonal_cluster,n_candidates,n_selected,n_orthogonal_diversifiers,avg_health_score,max_health_score,avg_abs_corr_within_cluster,run_id,diversity_version
0,volatility_structure,1,1,1,62.0,62.0,None,phase2_signal_diversity_20260511_172132,phase2_signal_diversity_v1


[03G_PROFILE] SQLite readback: diversity outputs | elapsed_seconds=0.029 | memory_before_mb=643.2 | memory_after_mb=644.4 | memory_delta_mb=1.2 | selection_readback_rows=1 | diagnostics_readback_rows=1 | family_readback_rows=1 | cluster_readback_rows=1


In [12]:
with profile_block("SQLite readback: candidate diagnostics") as profile_metrics:
    from src.db import load_table

    meta = load_table("candidate_signal_metadata_current")
    quality = load_table("candidate_signal_quality_current")
    scores = load_table("signal_scores_current")
    health = load_table("signal_health_score_current")
    div = load_table("signal_diversity_selection_current")

    new = ["volume_acceleration_20", "price_volume_divergence_20"]
    diagnostic_signal_names = (
        pd.Series(list(eligible_signal_names) + new, dtype="object")
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )
    signals = load_candidate_signals_by_names(
        diagnostic_signal_names,
        current=True,
        db_path=DB_PATH,
        chunksize=500_000,
    )

    placeholders = ",".join("?" for _ in diagnostic_signal_names)
    with sqlite3.connect(DB_PATH) as conn:
        signal_row_counts = readback_sql(
            f"""
            SELECT
                signal_name,
                COUNT(signal_value) AS count,
                COUNT(*) AS size,
                AVG(signal_value) AS mean,
                MIN(signal_value) AS min,
                MAX(signal_value) AS max,
                COUNT(DISTINCT Date) AS n_dates,
                COUNT(DISTINCT ticker) AS n_tickers
            FROM candidate_signals_current
            WHERE signal_name IN ({placeholders})
            GROUP BY signal_name
            ORDER BY signal_name
            """,
            conn,
            params=diagnostic_signal_names,
        ) if diagnostic_signal_names else pd.DataFrame()
        signal_preview = readback_sql(
            f"""
            SELECT Date, ticker, signal_name, signal_value, run_id, signal_version
            FROM candidate_signals_current
            WHERE signal_name IN ({placeholders})
            ORDER BY signal_name, Date, ticker
            LIMIT 20
            """,
            conn,
            params=diagnostic_signal_names,
        ) if diagnostic_signal_names else pd.DataFrame()

    profile_metrics["metadata_rows_loaded"] = len(meta)
    profile_metrics["quality_rows_loaded"] = len(quality)
    profile_metrics["candidate_signal_rows_loaded"] = len(signals)
    profile_metrics["candidate_signal_names_loaded"] = len(diagnostic_signal_names)
    profile_metrics["signal_row_count_rows"] = len(signal_row_counts)
    profile_metrics["signal_preview_rows"] = len(signal_preview)
    profile_metrics["score_rows_loaded"] = len(scores)
    profile_metrics["health_rows_loaded"] = len(health)
    profile_metrics["diversity_rows_loaded"] = len(div)

    print("Metadata rows:")
    display(meta[meta["signal_name"].isin(diagnostic_signal_names)].head(20))

    print("Quality rows:")
    display(quality[quality["signal_name"].isin(diagnostic_signal_names)].head(20))

    print("Signal row counts / non-null:")
    display(signal_row_counts)

    print("Signal preview rows:")
    display(signal_preview)

    print("Scoring rows:")
    display(scores[scores["signal_name"].isin(diagnostic_signal_names)].sort_values(["signal_name", "horizon"]).head(20))

    print("Health rows:")
    display(health[health["signal_name"].isin(diagnostic_signal_names)].sort_values(["signal_name", "horizon"]).head(20))

    print("Diversity rows:")
    display(div[div["signal_name"].isin(diagnostic_signal_names)].head(20))


load_candidate_signals_by_names chunk 1: signals=['price_volume_divergence_20'], rows=500,000, Date nulls=0


load_candidate_signals_by_names chunk 2: signals=['price_volume_divergence_20'], rows=500,000, Date nulls=0


load_candidate_signals_by_names chunk 3: signals=['price_volume_divergence_20', 'vol_of_vol_20'], rows=500,000, Date nulls=0


load_candidate_signals_by_names chunk 4: signals=['vol_of_vol_20'], rows=500,000, Date nulls=0


load_candidate_signals_by_names chunk 5: signals=['vol_of_vol_20', 'volume_acceleration_20'], rows=500,000, Date nulls=0


...

Health rows:


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,decay_risk_flag,mean_rolling_ic,recent_ic,early_ic,ic_change,sign_stability,adjusted_best_abs_ic,recommended_use,regime_fragility_flag,regime_consistency_score,regime_sample_weight,wfv_status,direction_flip_warning,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,signal_health_score,signal_health_gate,health_notes,run_id,health_version
59,vol_of_vol_20,1,volatility_structure,POSITIVE_EDGE,NO_SIGNAL,0.015976,0.015976,0.128788,REJECTED_LOW_SIGNAL,UNSTABLE,MODERATE_DECAY_RISK,0.000247,0.001534,0.005166,-0.003632,0.542132,0.012431,WATCHLIST,HIGH_REGIME_FRAGILITY,0.333333,0.936667,MISSING_WFV,0,None,None,None,14.0,REJECTED_RESEARCH,Insufficient combined evidence; low scoring si...,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1
18,vol_of_vol_20,5,volatility_structure,POSITIVE_EDGE,NO_SIGNAL,0.015976,0.015976,0.128788,REJECTED_LOW_SIGNAL,UNSTABLE,MODERATE_DECAY_RISK,0.006672,0.008072,0.015475,-0.007403,0.576297,0.048837,CONDITIONAL,HIGH_REGIME_FRAGILITY,0.666667,0.936667,MISSING_WFV,0,None,None,None,37.0,REJECTED_RESEARCH,Insufficient combined evidence; low scoring si...,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1
0,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,0.015976,0.015976,0.128788,WATCHLIST,STABLE,LOW_DECAY_RISK,0.012321,0.013839,0.026385,-0.012547,0.619072,0.062676,CONDITIONAL,HIGH_REGIME_FRAGILITY,0.666667,0.936667,MISSING_WFV,0,None,None,None,62.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1
3,vol_of_vol_20,20,volatility_structure,POSITIVE_EDGE,WEAK,0.015976,0.015976,0.128788,WATCHLIST,STABLE,MODERATE_DECAY_RISK,0.015693,0.020585,0.047790,-0.027205,0.615069,0.079858,CONDITIONAL,HIGH_REGIME_FRAGILITY,0.666667,0.936667,MISSING_WFV,0,None,None,None,54.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1


Diversity rows:


,signal_key,signal_name,horizon,signal_family,diversity_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,repro_candidate_tier,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group,run_id,diversity_version
0,vol_of_vol_20__h10,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_DIVERSIFIER,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,62.0,WATCHLIST_ALPHA_RESEARCH,CONDITIONAL_PASS,0.857143,0.015966,1,1,0.0,Selected within correlation threshold 0.85.,ORTHOGONAL_DIVERSIFIER_SELECTED,phase2_signal_diversity_20260511_172132,phase2_signal_diversity_v1


[03G_PROFILE] SQLite readback: candidate diagnostics | elapsed_seconds=5.913 | memory_before_mb=644.6 | memory_after_mb=1794.6 | memory_delta_mb=1150.0 | metadata_rows_loaded=108 | quality_rows_loaded=108 | candidate_signal_rows_loaded=3008532 | candidate_signal_names_loaded=3 | signal_row_count_rows=3 | signal_preview_rows=20 | score_rows_loaded=92 | health_rows_loaded=92 | diversity_rows_loaded=1


In [13]:
with profile_block("SQLite readback: similarity diagnostics") as profile_metrics:
    sim = load_table("signal_diversity_similarity_current")

    profile_metrics["similarity_readback_rows"] = len(sim)

    display(
        sim[
            sim["signal_name_1"].isin(diagnostic_signal_names) | sim["signal_name_2"].isin(diagnostic_signal_names)
        ]
        .sort_values("correlation", key=lambda s: s.abs(), ascending=False)
        .head(20)
    )


,signal_key_1,signal_key_2,signal_name_1,horizon_1,signal_name_2,horizon_2,correlation,run_id,diversity_version
0,vol_of_vol_20__h10,vol_of_vol_20__h10,vol_of_vol_20,10,vol_of_vol_20,10,1.0,phase2_signal_diversity_20260511_172132,phase2_signal_diversity_v1


[03G_PROFILE] SQLite readback: similarity diagnostics | elapsed_seconds=0.008 | memory_before_mb=1794.6 | memory_after_mb=1795.4 | memory_delta_mb=0.8 | similarity_readback_rows=1
